# 04 · Fine-tuned MiniLM (Prompt 8), hybrid RRF (Prompt 9), cross-encoder rerank (Prompt 10), owner: M2

Before running:
1. **Accelerator: GPU T4 x1**. **Internet: On**.
2. **Add Input** (Your Work):
   - `fever-corpus-v1` (Datasets)
   - the output of notebook **02 · BM25** (or the dataset made from it, e.g. `bm25-baseline`): supplies
     `outputs/bm25/run_train.jsonl`, `run_val.jsonl` and `run_test.jsonl`
   - optional: the output of notebook **03 · Dense**. Its `dense_offshelf/metrics.json` gives the "before"
     numbers; without it the script re-evaluates the off-the-shelf model (about 2 extra minutes).

Expected GPU time: about 30-40 minutes (two short fine-tuning runs, three corpus encodings of about 1 minute
each, and the cross-encoder rerank of 50 sentences for each of 22k claims, about 10-15 minutes).

When it finishes: **Save Version → Save & Run All (Commit)**. Notebooks 05 and 06 need this output
(`models/minilm_ft`, `outputs/dense_ft`, `outputs/hybrid`, `outputs/rerank`).

In [ ]:
import os, subprocess
REPO_URL = "https://github.com/group-mauve/claim-verification-pipeline.git"   # public repo
BRANCH = "main"
WORK = "/kaggle/working/project"

def git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    print((r.stdout + r.stderr).strip())
    if r.returncode != 0:
        raise RuntimeError("git failed (see message above). Is Internet on in the notebook settings?")

if not os.path.exists(WORK):
    git("clone", "--depth", "1", "--branch", BRANCH, REPO_URL, WORK)
else:
    git("-C", WORK, "pull", "origin", BRANCH)   # pick up teammates' latest pushes
os.chdir(WORK)
git("log", "--oneline", "-1")                    # which commit this run uses
print(os.getcwd(), sorted(os.listdir()))

In [ ]:
!pip install -q faiss-cpu sentence-transformers datasets pyyaml pytest
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Which inputs are attached? Every folder this notebook needs should appear in this list.
import glob
found = sorted(p for p in glob.glob("/kaggle/input/**/*", recursive=True)
               if p.endswith(("sentences.parquet", "metrics.json", "minilm_ft/config.json", "hotpot_dev_distractor_v1.json")))
print("\n".join(found) if found else "Nothing attached: use Add Input (see the first cell)")

## Unit tests

In [ ]:
!python -m pytest -q -rs

## Prompt 8 · Fine-tune MiniLM with BM25 hard negatives (seed 42)

Builds one (claim, evidence sentence, BM25 hard negative) row per SUPPORTS / REFUTES train claim, prints the
false-negative check, trains 1 epoch with MultipleNegativesRankingLoss, then re-encodes the corpus and
evaluates exactly like Prompt 6.

In [ ]:
!python -m src.retrieval.finetune --config configs/finetune.yaml

In [ ]:
import json, pandas as pd
s = json.load(open("outputs/finetune/metrics.json"))
print(f"{s['n_rows']:,} training rows {s['row_stats']} | false-negative flag rate {s['false_negative_share']:.1%}")
rows = {}
for name, key in (("off the shelf", "before"), ("fine-tuned", "after")):
    for sp in ("val", "test"):
        rep = s[f"{sp}_{key}"]
        if rep:
            rows[(name, sp)] = {k: v for k, v in rep["sentence"].items() if not k.startswith("evidence_set")}
pd.DataFrame(rows).T.round(4)

In [ ]:
fn = json.load(open("outputs/finetune/false_negative_check.json"))
pd.set_option("display.max_colwidth", 120)
pd.DataFrame(fn["examples"])[["label", "anchor", "positive", "negative", "sim_positive", "sim_negative", "flagged"]]

## Second seed (43), only to measure the noise between two seeds

Prompt 10 keeps the reranker only if its gain is larger than this noise. The model and embeddings of this run
are deleted at the end of the notebook; its run files and metrics are kept.

In [ ]:
!python -m src.retrieval.finetune --config configs/finetune.yaml --seed 43

## Prompt 9 · Hybrid retrieval: RRF and weighted score fusion

In [ ]:
!python -m src.retrieval.hybrid --config configs/hybrid.yaml

In [ ]:
from IPython.display import Markdown
Markdown(open("outputs/hybrid/table.md").read())

## Prompt 10 · Cross-encoder rerank of the hybrid top 50

In [ ]:
!python -m src.retrieval.rerank --config configs/rerank.yaml

In [ ]:
m = json.load(open("outputs/rerank/metrics.json"))
print(m["decision"])
Markdown(open("outputs/rerank/table.md").read())

## Keep the output small

Deletes the second-seed model and embeddings (only its run files are needed), training checkpoints, and
off-the-shelf embeddings recomputed in this session (notebook 03 already has them).

In [ ]:
import shutil
for p in ["models/minilm_ft_seed43", "outputs/dense_ft_seed43/embeddings", "outputs/finetune/checkpoints",
          "outputs/finetune_seed43/checkpoints", "outputs/dense_offshelf/embeddings", "outputs/dense_offshelf/index.faiss"]:
    if os.path.isdir(p):
        shutil.rmtree(p)
    elif os.path.exists(p):
        os.remove(p)
!du -sh models/* outputs/*